In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
import json
import sys
import os
import pickle
sys.path.append("../../../src/experiment_util") 
import experiment_utils

In [8]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]

sampled_matched_df = pd.read_pickle(dataframes_path + "/sampled_matched_perturbed_df_final.pkl")
first_n_df = pd.read_pickle(dataframes_path + "/first_n_df.pkl")
irl_policies = pd.read_pickle(dataframes_path + "/first_n_irl.pkl")

In [9]:
users_unique = sampled_matched_df.drop_duplicates(subset='user', keep='first')
irl_policies = irl_policies.merge(users_unique[['user', 'russian']], on='user', how='left')


In [ ]:
misclassified_users_rf_no_agree_all_subs_dict = {}
misclassified_users_xgb_no_agree_all_subs_dict = {}


irl_policies_results = []

for n in irl_policies.n.unique():

    running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
    running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

    f1_list_rf = []
    f1_list_xgb = []

    misclassified_users_rf_all_subs = defaultdict(int)
    misclassified_users_xgb_all_subs = defaultdict(int)
    for run in irl_policies.run.unique():
        
        print(n,"run:",run)
        seed = 100 + run


        df_russian_sampled = irl_policies[(irl_policies.run == run) & (irl_policies.n == n) & (irl_policies.russian == 1)].copy()
        df_sampled = irl_policies[(irl_policies.run == run) & (irl_policies.n == n) & (irl_policies.russian == 0)].copy()

        df_russian_sampled["feature_col"] = df_russian_sampled['policy'].apply(lambda x: x.numpy().flatten())
        df_sampled["feature_col"] = df_sampled['policy'].apply(lambda x: x.numpy().flatten())

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_russian_sampled,df_sampled,seed = seed)

        f1_list_rf.append(overall_rf_f1)
        f1_list_xgb.append(overall_xgb_f1)
        running_confusion_matrix_rf += confusion_matrix_rf
        running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]

    print("n", n)
    print("Final Confusion Matrix (Random Forest):\n", running_confusion_matrix_rf)    
    f1 = np.mean(f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Confusion Matrix (XGBoost):\n", running_confusion_matrix_xgb)        
    f1 = np.mean(f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")


    results = {
        "n":n,
        "f1_list_rf":f1_list_rf,
        "f1_list_xgb":f1_list_xgb,
        "running_confusion_matrix_rf":running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":misclassified_users_xgb_all_subs
    }

    misclassified_users_rf_no_agree_all_subs_dict[n] = misclassified_users_rf_all_subs
    misclassified_users_xgb_no_agree_all_subs_dict[n] = misclassified_users_xgb_all_subs

    irl_policies_results.append(results)

output_dir = project_output_path + "/early_detection"
os.makedirs(output_dir, exist_ok=True)
with open(output_dir +'/irl_policies_results.pkl', 'wb') as f:
    pickle.dump(irl_policies_results, f)



-1 run: 0


NameError: name 'experiment_utils' is not defined